# Hairdresser / Barber Intent Dataset

Gera um dataset sintético para classificação de intenção em português do Brasil, seguindo a mesma escala do dataset `legal` deste repositório: **1.234 exemplos de treino por intenção e 13 de avaliação por intenção**.

Pipeline: `human request → intent → BehaviorID → Skill`.

O notebook também valida distribuição, unicidade, schema e sobreposição entre treino e avaliação.

In [ ]:
from itertools import product
from pathlib import Path
import json
import pandas as pd

OUT = Path('../datasets/hairdresser')
OUT.mkdir(parents=True, exist_ok=True)
DOMAIN = 'hairdresser'
TRAIN_PER_LABEL = 1234
EVAL_PER_LABEL = 13


## Taxonomia

As intenções são deliberadamente sobre **o que a pessoa pretende**, e não sobre como o sistema deve executar a ação.

In [ ]:
LABELS = [
    'AGENDAMENTO', 'CANCELAMENTO', 'REAGENDAMENTO', 'DISPONIBILIDADE',
    'SERVICO', 'PRECO', 'PROFISSIONAL', 'HORARIO', 'ENDERECO',
    'PREFERENCIA', 'CUIDADOS', 'PAGAMENTO'
]
len(LABELS), len(LABELS) * TRAIN_PER_LABEL, len(LABELS) * EVAL_PER_LABEL

In [ ]:
# Os pools abaixo são a fonte de variação lexical usada na geração.
# Para manter o notebook compacto e reproduzível, cada intenção combina
# quatro dimensões de linguagem em um espaço de 8^3*8 variações.
POOLS = {
  "AGENDAMENTO": {
    "A": [
      "Quero",
      "Gostaria de",
      "Preciso",
      "Pode marcar",
      "Queria marcar",
      "Tem como marcar",
      "Me ajuda a marcar",
      "Quero deixar marcado"
    ],
    "B": [
      "um corte",
      "um horário para corte",
      "um corte de cabelo",
      "um corte masculino",
      "um corte feminino",
      "um atendimento no salão",
      "um horário com o barbeiro",
      "um horário com a cabeleireira"
    ],
    "C": [
      "para esta semana",
      "para a próxima semana",
      "para amanhã",
      "para sábado",
      "para depois do trabalho",
      "para a tarde",
      "para a manhã",
      "para o fim do dia"
    ],
    "D": [
      "por favor",
      "se possível",
      "para mim",
      "sem complicação"
    ]
  },
  "CANCELAMENTO": {
    "A": [
      "Quero cancelar",
      "Preciso cancelar",
      "Gostaria de cancelar",
      "Pode cancelar",
      "Queria desmarcar",
      "Preciso desmarcar",
      "Dá para cancelar",
      "Quero retirar"
    ],
    "B": [
      "meu horário",
      "meu agendamento",
      "a marcação",
      "o corte marcado",
      "o atendimento",
      "a reserva",
      "o horário com o barbeiro",
      "o horário com a cabeleireira"
    ],
    "C": [
      "de hoje",
      "de amanhã",
      "desta semana",
      "de sábado",
      "que fiz para mim",
      "que marquei no salão",
      "que marquei para o corte",
      "que está agendado"
    ],
    "D": [
      "por favor",
      "se possível",
      "sem remarcar agora",
      "porque não vou conseguir ir"
    ]
  },
  "REAGENDAMENTO": {
    "A": [
      "Quero remarcar",
      "Preciso remarcar",
      "Gostaria de remarcar",
      "Pode mudar",
      "Queria mudar",
      "Preciso trocar",
      "Dá para alterar",
      "Quero transferir"
    ],
    "B": [
      "meu horário",
      "meu agendamento",
      "a marcação",
      "o corte marcado",
      "o atendimento",
      "a reserva",
      "o horário com o barbeiro",
      "o horário com a cabeleireira"
    ],
    "C": [
      "para amanhã",
      "para outro dia",
      "para a próxima semana",
      "para mais tarde",
      "para a parte da manhã",
      "para a parte da tarde",
      "para sábado",
      "para um horário diferente"
    ],
    "D": [
      "por favor",
      "se tiver disponibilidade",
      "sem cancelar o serviço",
      "porque surgiu um compromisso"
    ]
  },
  "DISPONIBILIDADE": {
    "A": [
      "Tem",
      "Existe",
      "Há",
      "Vocês têm",
      "Consigo encontrar",
      "Dá para conseguir",
      "Queria saber se tem",
      "Quero verificar se existe"
    ],
    "B": [
      "horário para corte",
      "vaga para corte",
      "horário com barbeiro",
      "horário com cabeleireira",
      "vaga para atendimento",
      "horário para barba",
      "vaga para coloração",
      "horário para hoje"
    ],
    "C": [
      "hoje",
      "amanhã",
      "nesta semana",
      "no sábado",
      "pela manhã",
      "à tarde",
      "depois das seis",
      "no fim de semana"
    ],
    "D": [
      "por favor",
      "para uma pessoa",
      "para mim",
      "sem preferência de profissional"
    ]
  },
  "SERVICO": {
    "A": [
      "Quero saber sobre",
      "Gostaria de conhecer",
      "Preciso entender",
      "Queria informações sobre",
      "Vocês fazem",
      "Quais opções existem para",
      "Tenho interesse em",
      "Pode me explicar"
    ],
    "B": [
      "corte de cabelo",
      "corte masculino",
      "corte feminino",
      "barba",
      "coloração",
      "mechas",
      "escova",
      "tratamento capilar"
    ],
    "C": [
      "no salão",
      "com um profissional",
      "para cabelo curto",
      "para cabelo comprido",
      "para um visual novo",
      "antes de marcar",
      "para o meu tipo de cabelo",
      "na próxima visita"
    ],
    "D": [
      "por favor",
      "e como funciona",
      "e qual é a duração",
      "e se precisa agendar"
    ]
  },
  "PRECO": {
    "A": [
      "Quanto custa",
      "Qual o valor de",
      "Gostaria de saber o preço de",
      "Quero saber quanto fica",
      "Pode me passar o preço de",
      "Você sabe o valor de",
      "Preciso saber quanto custa",
      "Tem como informar o preço de"
    ],
    "B": [
      "um corte",
      "um corte masculino",
      "um corte feminino",
      "uma barba",
      "uma coloração",
      "umas mechas",
      "uma escova",
      "um tratamento capilar"
    ],
    "C": [
      "no salão",
      "para cabelo curto",
      "para cabelo comprido",
      "com profissional sênior",
      "para este mês",
      "antes de agendar",
      "incluindo lavagem",
      "sem outros serviços"
    ],
    "D": [
      "por favor",
      "hoje",
      "para mim",
      "em média"
    ]
  },
  "PROFISSIONAL": {
    "A": [
      "Quero marcar com",
      "Gostaria de atendimento com",
      "Prefiro fazer com",
      "Quero saber quem atende como",
      "Preciso encontrar",
      "Tem como escolher",
      "Queria ser atendido por",
      "Gostaria de conhecer"
    ],
    "B": [
      "um barbeiro",
      "uma cabeleireira",
      "um profissional específico",
      "quem faz cortes masculinos",
      "quem faz coloração",
      "quem faz mechas",
      "quem entende de cabelo cacheado",
      "quem trabalha com barba"
    ],
    "C": [
      "no salão",
      "nesta semana",
      "no sábado",
      "pela manhã",
      "à tarde",
      "para o meu próximo corte",
      "para uma mudança de visual",
      "para um atendimento específico"
    ],
    "D": [
      "por favor",
      "se houver disponibilidade",
      "se possível",
      "antes de agendar"
    ]
  },
  "HORARIO": {
    "A": [
      "Quero saber o horário",
      "Gostaria de saber o horário",
      "Que horas vocês atendem",
      "Qual o horário de atendimento",
      "Preciso consultar o horário",
      "Pode me informar quando vocês atendem",
      "Queria confirmar o horário",
      "Vocês atendem"
    ],
    "B": [
      "do salão",
      "da barbearia",
      "no sábado",
      "no domingo",
      "durante a semana",
      "à noite",
      "pela manhã",
      "à tarde"
    ],
    "C": [
      "hoje",
      "amanhã",
      "nesta semana",
      "no feriado",
      "para corte",
      "para barba",
      "para coloração",
      "para atendimento"
    ],
    "D": [
      "por favor",
      "antes de ir",
      "para eu me organizar",
      "se estiver aberto"
    ]
  },
  "ENDERECO": {
    "A": [
      "Onde fica",
      "Qual é o endereço de",
      "Como chego ao",
      "Pode me passar o endereço de",
      "Quero localizar",
      "Preciso saber onde fica",
      "Vocês ficam",
      "Qual a localização de"
    ],
    "B": [
      "o salão",
      "a barbearia",
      "o estabelecimento",
      "a unidade",
      "o espaço de atendimento",
      "o salão de vocês",
      "a barbearia de vocês",
      "o local"
    ],
    "C": [
      "no centro",
      "perto do centro",
      "na cidade",
      "para eu ir hoje",
      "antes do meu horário",
      "para o meu agendamento",
      "de carro",
      "a pé"
    ],
    "D": [
      "por favor",
      "se possível",
      "para eu me localizar",
      "e qual é o ponto de referência"
    ]
  },
  "PREFERENCIA": {
    "A": [
      "Quero",
      "Gostaria de",
      "Prefiro",
      "Preciso de",
      "Queria",
      "Tenho preferência por",
      "Posso escolher",
      "Quero manter"
    ],
    "B": [
      "corte mais curto",
      "corte na tesoura",
      "corte na máquina",
      "degradê",
      "barba mais aparada",
      "uma cor específica",
      "um profissional específico",
      "um horário específico"
    ],
    "C": [
      "no meu próximo atendimento",
      "quando eu agendar",
      "para o próximo corte",
      "para a próxima visita",
      "sempre que eu marcar",
      "no atendimento desta semana",
      "na próxima vez",
      "quando houver vaga"
    ],
    "D": [
      "por favor",
      "se possível",
      "sem trocar a preferência",
      "se vocês puderem"
    ]
  },
  "CUIDADOS": {
    "A": [
      "Quero saber como cuidar",
      "Gostaria de orientação para cuidar",
      "Preciso saber como tratar",
      "Queria dicas para cuidar",
      "Como faço para manter",
      "Pode me orientar sobre como cuidar",
      "Quero entender como manter",
      "Preciso de recomendações para"
    ],
    "B": [
      "meu corte",
      "meu cabelo",
      "meu cabelo cacheado",
      "meu cabelo colorido",
      "as mechas",
      "a barba",
      "o couro cabeludo",
      "o resultado do tratamento"
    ],
    "C": [
      "em casa",
      "depois do atendimento",
      "entre os cortes",
      "durante a semana",
      "sem estragar o resultado",
      "depois da coloração",
      "até a próxima visita",
      "para manter o visual"
    ],
    "D": [
      "por favor",
      "com segurança",
      "por mais tempo",
      "sem gastar muito"
    ]
  },
  "PAGAMENTO": {
    "A": [
      "Quero saber se posso pagar",
      "Gostaria de saber se aceitam",
      "Preciso saber como pagar",
      "Quais formas de pagamento vocês aceitam",
      "Posso pagar",
      "Dá para pagar",
      "Queria confirmar o pagamento de",
      "Vocês recebem"
    ],
    "B": [
      "o corte",
      "a barba",
      "o atendimento",
      "a coloração",
      "o serviço",
      "o agendamento",
      "o tratamento",
      "o valor"
    ],
    "C": [
      "com Pix",
      "com cartão",
      "com dinheiro",
      "na hora",
      "antes do atendimento",
      "depois do serviço",
      "pelo celular",
      "no salão"
    ],
    "D": [
      "por favor",
      "hoje",
      "no meu próximo atendimento",
      "sem taxa adicional"
    ]
  }
}

In [ ]:
def generate(label, start, count):
    p = POOLS[label]
    A, B, C, D = p['A'], p['B'], p['C'], p['D']
    per_a = len(B) * len(C) * len(D)
    total = len(B) * len(C) * len(D)
    rows = []
    k = start
    while len(rows) < count:
        ia = (k // total) % len(A)
        rem = k % total
        ib = (rem // (len(C) * len(D))) % len(B)
        ic = (rem // len(D)) % len(C)
        id_ = rem % len(D)
        text = f'{A[ia]} {B[ib]} {C[ic]} {D[id_]}'
        if k % 4 == 0: text += '.'
        elif k % 4 == 1: text += '?'
        elif k % 4 == 2: text += ', por favor.'
        else: text += '!'
        rows.append({'text': text, 'label': label, 'domain': DOMAIN})
        k += 1
    return rows

train, evaluation = [], []
for label in LABELS:
    evaluation += generate(label, 0, EVAL_PER_LABEL)
    train += generate(label, EVAL_PER_LABEL, TRAIN_PER_LABEL)

len(train), len(evaluation)

In [ ]:
train_path = OUT / 'train.jsonl'
eval_path = OUT / 'eval.jsonl'

train_path.write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in train) + '\n', encoding='utf-8')
eval_path.write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in evaluation) + '\n', encoding='utf-8')

print(train_path, len(train))
print(eval_path, len(evaluation))

In [ ]:
train_df = pd.DataFrame(train)
eval_df = pd.DataFrame(evaluation)

assert len(train_df) == 14808
assert len(eval_df) == 156
assert train_df.groupby('label').size().eq(1234).all()
assert eval_df.groupby('label').size().eq(13).all()
assert train_df['text'].is_unique
assert eval_df['text'].is_unique
assert not set(train_df['text']).intersection(eval_df['text'])
assert set(train_df['label']) == set(LABELS)
assert set(eval_df['label']) == set(LABELS)

print('Validation OK')
display(train_df.groupby('label').size().rename('train').to_frame())

In [ ]:
display(train_df.sample(20, random_state=42))
display(eval_df.groupby('label').head(2))